In [75]:
import os
import json
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from pathlib import Path
from c3ppi.ppi_predictor.models.C3BCA import PPI_Model

In [76]:
bac_info_file = "./data/bac_info.json"
per_prot_dir = "./data/inference_results/per_protein_results"
per_bac_dir = "./data/inference_results/per_bac_results"

os.makedirs(per_bac_dir, exist_ok=True)

## Interactions per bac - Score = 1

In [77]:
'''import duckdb
import pandas as pd
import json

# Load bac_info.json
with open(bac_info_file, 'r') as f:
    bac_info = json.load(f)

# Connect DuckDB
con = duckdb.connect()

# Loop over bac
for bac, info in bac_info.items():
    output_file = Path(per_bac_dir) / f"{bac}_interactions.parquet"
    
    if not output_file.exists():

        prot_list = []
        for up, up_prot_list in info.items():
            prot_list.extend(up_prot_list)
        prot_list = list(set(prot_list))

        if len(prot_list) < 20000:

            # Convert list to DataFrame
            prot_df = pd.DataFrame({"prot": prot_list})

            # Register as temp table
            con.register("prot_table", prot_df)

            # Query with JOIN
            query = """
                SELECT p.*
                FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet') p
                INNER JOIN prot_table t
                ON p.bacterial_protein = t.prot
                WHERE p.score = 1
            """

            df = con.execute(query).df()
            print(f"{bac}: {len(df)} rows")
            
            # Save to parquet
            df.to_parquet(output_file, index=False)

    else:
        print(f"{bac}: File already exists, skipping...")'''


'import duckdb\nimport pandas as pd\nimport json\n\n# Load bac_info.json\nwith open(bac_info_file, \'r\') as f:\n    bac_info = json.load(f)\n\n# Connect DuckDB\ncon = duckdb.connect()\n\n# Loop over bac\nfor bac, info in bac_info.items():\n    output_file = Path(per_bac_dir) / f"{bac}_interactions.parquet"\n\n    if not output_file.exists():\n\n        prot_list = []\n        for up, up_prot_list in info.items():\n            prot_list.extend(up_prot_list)\n        prot_list = list(set(prot_list))\n\n        if len(prot_list) < 20000:\n\n            # Convert list to DataFrame\n            prot_df = pd.DataFrame({"prot": prot_list})\n\n            # Register as temp table\n            con.register("prot_table", prot_df)\n\n            # Query with JOIN\n            query = """\n                SELECT p.*\n                FROM parquet_scan(\'./data/inference_results/per_protein_results/*.parquet\') p\n                INNER JOIN prot_table t\n                ON p.bacterial_protein = t.p

In [78]:
import duckdb
import pandas as pd
import json

# Load bac_info.json
with open(bac_info_file, 'r') as f:
    bac_info = json.load(f)

# Connect DuckDB
con = duckdb.connect()
batch_size = 20000

for bac, info in bac_info.items():
    output_file = Path(per_bac_dir) / f"{bac}_interactions.parquet"

    if output_file.exists():
        print(f"{bac}: File already exists, skipping...")
        continue

    # Flatten and deduplicate protein list
    prot_list = list({prot for up_list in info.values() for prot in up_list})

    if not prot_list:
        print(f"{bac}: No proteins found, skipping...")
        continue

    for i in range(0, len(prot_list), batch_size):
        prot_chunk = prot_list[i:i + batch_size]
        prot_df = pd.DataFrame({"prot": prot_chunk})
        con.register("prot_table", prot_df)

        batch_file = Path(per_bac_dir) / f"{bac}_{i}_interactions.parquet"

        if batch_file.exists():
            print(f"{bac}: Batch file already exists, skipping...")
            continue

        query = """
            SELECT p.*
            FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet') p
            INNER JOIN prot_table t
            ON p.bacterial_protein = t.prot
            WHERE p.score = 1
            """
        chunk_df = con.execute(query).df()

        if len(chunk_df) > 0:
            chunk_df.to_parquet(batch_file, index=False)
            print(f"Saved batch {i} for {bac} with {len(chunk_df)} rows")
        else:
            print(f"No interactions found for batch {i} for {bac}")



g__Alloscardovia: File already exists, skipping...
g__Arcanobacterium: File already exists, skipping...
g__Brevibacterium: File already exists, skipping...
g__CAG-352: File already exists, skipping...
g__Christensenellaceae R-7 group: File already exists, skipping...
g__Clostridium innocuum group: File already exists, skipping...
g__Colidextribacter: File already exists, skipping...
g__Corynebacterium: File already exists, skipping...
g__Cutibacterium: File already exists, skipping...
g__Dermabacter: File already exists, skipping...
g__Desulfovibrio: File already exists, skipping...
g__Dialister: File already exists, skipping...
g__Dorea: File already exists, skipping...
g_Shigella: File already exists, skipping...
g__Eubacterium hallii group: File already exists, skipping...
g__Eubacterium nodatum group: File already exists, skipping...
g__Facklamia: File already exists, skipping...
g__Family XIII AD3011 group: File already exists, skipping...
g__Filifactor: File already exists, skipp

In [79]:
import duckdb

con = duckdb.connect()

query = """
    SELECT COUNT(*) AS total_records
    FROM parquet_scan('./data/inference_results/per_bac_results/*.parquet')
"""

result = con.execute(query).fetchone()[0]
print(f"Total records: {result}")

# 1481068857/8922520656 = 0.16

Total records: 1481068857


In [80]:
'''pheno_file = "./data/bacterial_phenotype_assignment.csv"

pheno_df = pd.read_csv(pheno_file)
pheno_df = pheno_df[["Taxonomy", "Phenotype_Assignment"]]

for ph in ["Healthy-associated", "CRC-associated"]:
    ass_list = [ph, "Both (No clear association)"]

    ph_df = pheno_df[pheno_df["Phenotype_Assignment"].isin(ass_list)]

    pheno_bac_files = []

    for bac in ph_df["Taxonomy"].unique():
        # Take recursive glob all files starts with bac
        pheno_bac_files.extend(Path(per_bac_dir).glob(f"{bac}*.parquet"))
    
    ph_interactions_file = "./data/{}_interactions.txt".format(ph)
    no_int = 0
    with open(ph_interactions_file, "w") as fout:
        for bac in pheno_bac_files:
            df = pd.read_parquet(bac, engine="pyarrow")

            # Vectorized write (faster than iterrows)
            lines = df[["bacterial_protein", "human_protein"]].astype(str).agg("\t".join, axis=1)
            fout.write("\n".join(lines) + "\n")

            no_int += len(df)

    print(f"{ph}: {no_int} interactions written to {ph_interactions_file}")'''
    

'pheno_file = "./data/bacterial_phenotype_assignment.csv"\n\npheno_df = pd.read_csv(pheno_file)\npheno_df = pheno_df[["Taxonomy", "Phenotype_Assignment"]]\n\nfor ph in ["Healthy-associated", "CRC-associated"]:\n    ass_list = [ph, "Both (No clear association)"]\n\n    ph_df = pheno_df[pheno_df["Phenotype_Assignment"].isin(ass_list)]\n\n    pheno_bac_files = []\n\n    for bac in ph_df["Taxonomy"].unique():\n        # Take recursive glob all files starts with bac\n        pheno_bac_files.extend(Path(per_bac_dir).glob(f"{bac}*.parquet"))\n\n    ph_interactions_file = "./data/{}_interactions.txt".format(ph)\n    no_int = 0\n    with open(ph_interactions_file, "w") as fout:\n        for bac in pheno_bac_files:\n            df = pd.read_parquet(bac, engine="pyarrow")\n\n            # Vectorized write (faster than iterrows)\n            lines = df[["bacterial_protein", "human_protein"]].astype(str).agg("\t".join, axis=1)\n            fout.write("\n".join(lines) + "\n")\n\n            no_int +

In [81]:
'''import duckdb
import pandas as pd

con = duckdb.connect()

# 1. Degree of human proteins
query_human = """
    SELECT human_protein AS protein, COUNT(*) AS degree_human
    FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet')
    GROUP BY human_protein
"""

# 2. Degree of bacterial proteins
query_bacterial = """
    SELECT bacterial_protein AS protein, COUNT(*) AS degree_bacterial
    FROM parquet_scan('./data/inference_results/per_protein_results/*.parquet')
    GROUP BY bacterial_protein
"""

# Run queries
#df_human = con.execute(query_human).df()
df_bac = con.execute(query_bacterial).df()

# Optional: Save separately
#df_human.to_csv("./data/human_protein_degrees.csv", index=False)
df_bac.to_csv("./data/bacterial_protein_degrees.csv", index=False)'''

'import duckdb\nimport pandas as pd\n\ncon = duckdb.connect()\n\n# 1. Degree of human proteins\nquery_human = """\n    SELECT human_protein AS protein, COUNT(*) AS degree_human\n    FROM parquet_scan(\'./data/inference_results/per_protein_results/*.parquet\')\n    GROUP BY human_protein\n"""\n\n# 2. Degree of bacterial proteins\nquery_bacterial = """\n    SELECT bacterial_protein AS protein, COUNT(*) AS degree_bacterial\n    FROM parquet_scan(\'./data/inference_results/per_protein_results/*.parquet\')\n    GROUP BY bacterial_protein\n"""\n\n# Run queries\n#df_human = con.execute(query_human).df()\ndf_bac = con.execute(query_bacterial).df()\n\n# Optional: Save separately\n#df_human.to_csv("./data/human_protein_degrees.csv", index=False)\ndf_bac.to_csv("./data/bacterial_protein_degrees.csv", index=False)'

In [82]:
# Degree 
hum_degree_file = "./data/human_protein_degrees.csv"
bac_degree_file = "./data/bacterial_protein_degrees.csv"

In [83]:
hum_degree = pd.read_csv(hum_degree_file)
hum_degree.sort_values("degree_human", ascending=False).reset_index(drop=True).head(10)

,protein,degree_human
0,F7VJQ1,901198
1,Q9Y252,900024
2,O00165,899894
3,Q9H0V1,899554
4,Q9UHA4,899336
5,Q9UNL2,899020
6,P62993,898854
7,Q8WWA0,898807
8,Q9NVW2,898272
9,Q9NPJ8,897977


In [84]:
bac_degree = pd.read_csv(bac_degree_file)
bac_degree.sort_values("degree_bacterial", ascending=False).reset_index(drop=True).head(10)

,protein,degree_bacterial
0,N9V9L9,9782
1,K0WYI1,9782
2,A0A1H4H2Y1,9782
3,Q8RFD9,9782
4,A0A1H3ZQT9,9782
5,A0A1K1LL05,9782
6,S0IPY0,9782
7,A0A1Y3X3C3,9782
8,F4XF02,9782
9,C9KJF4,9782


In [85]:
'''crc_cooc_file = "./data/Healthy_Co_occurance_filtered.xlsx"
crc_cooc = pd.read_excel(crc_cooc_file, index_col=0)
common_no = list()
for _, row in crc_cooc.iterrows():
    common_human_proteins = {}

    bac1 = row["Bac1"]
    bac2 = row["Bac2"]

    bac1_files = list(Path(per_bac_dir).glob(f"*{bac1}*.parquet"))
    bac2_files = list(Path(per_bac_dir).glob(f"*{bac2}*.parquet"))

    if len(bac1_files) == 0 or len(bac2_files) == 0:
        common_no.append(None)
        continue

    # Extract human proteins for Bac1
    human_set_bac1 = set()
    for f in bac1_files:
        intf = pd.read_parquet(f, engine="pyarrow")
        human_set_bac1.update(intf["human_protein"])

    # Extract human proteins for Bac2
    human_set_bac2 = set()
    for f in bac2_files:
        intf = pd.read_parquet(f, engine="pyarrow")
        human_set_bac2.update(intf["human_protein"])

    # Find common human proteins
    common = human_set_bac1 & human_set_bac2
    if common:
        common_human_proteins[(bac1, bac2)] = common
        print(f"{bac1} & {bac2} → {len(common)} common human proteins")

    # Add the common number in a new row of the dataframe
    common_no.append(len(common) / 9864)

crc_cooc["NoHost"] = common_no

# Save to csv
crc_cooc.to_csv(crc_cooc_file.replace(".xlsx",".csv"))'''
        

'crc_cooc_file = "./data/Healthy_Co_occurance_filtered.xlsx"\ncrc_cooc = pd.read_excel(crc_cooc_file, index_col=0)\ncommon_no = list()\nfor _, row in crc_cooc.iterrows():\n    common_human_proteins = {}\n\n    bac1 = row["Bac1"]\n    bac2 = row["Bac2"]\n\n    bac1_files = list(Path(per_bac_dir).glob(f"*{bac1}*.parquet"))\n    bac2_files = list(Path(per_bac_dir).glob(f"*{bac2}*.parquet"))\n\n    if len(bac1_files) == 0 or len(bac2_files) == 0:\n        common_no.append(None)\n        continue\n\n    # Extract human proteins for Bac1\n    human_set_bac1 = set()\n    for f in bac1_files:\n        intf = pd.read_parquet(f, engine="pyarrow")\n        human_set_bac1.update(intf["human_protein"])\n\n    # Extract human proteins for Bac2\n    human_set_bac2 = set()\n    for f in bac2_files:\n        intf = pd.read_parquet(f, engine="pyarrow")\n        human_set_bac2.update(intf["human_protein"])\n\n    # Find common human proteins\n    common = human_set_bac1 & human_set_bac2\n    if common:\n

In [86]:
'''import pandas as pd

for ph in ["Healthy-associated", "CRC-associated"]:
    
    ph_interactions_file = "./data/{}_interactions.txt".format(ph)

    hum_prot_degree = {}
    bac_prot_degree = {}

    with open(ph_interactions_file, "r") as f:
        for line in f:
            line = line.strip().split("\t")
            if len(line) != 2:
                continue

            # ANAPODA!!!!!!!  
            hum_prot = line[1]
            bac_prot = line[0]

            if hum_prot not in hum_prot_degree:
                hum_prot_degree[hum_prot] = 1
            else:
                hum_prot_degree[hum_prot] += 1

            if bac_prot not in bac_prot_degree:
                bac_prot_degree[bac_prot] = 1
            else:
                bac_prot_degree[bac_prot] += 1

    # Save to csv
    hum_prot_degree_df = pd.DataFrame.from_dict(hum_prot_degree, orient="index", columns=["degree_human"])
    hum_prot_degree_df.sort_values("degree_human", ascending=False).reset_index().to_csv("./data/{}_human_protein_degrees.csv".format(ph), index=False)

    bac_prot_degree_df = pd.DataFrame.from_dict(bac_prot_degree, orient="index", columns=["degree_bacterial"])
    bac_prot_degree_df.sort_values("degree_bacterial", ascending=False).reset_index().to_csv("./data/{}_bacterial_protein_degrees.csv".format(ph), index=False)'''

'import pandas as pd\n\nfor ph in ["Healthy-associated", "CRC-associated"]:\n\n    ph_interactions_file = "./data/{}_interactions.txt".format(ph)\n\n    hum_prot_degree = {}\n    bac_prot_degree = {}\n\n    with open(ph_interactions_file, "r") as f:\n        for line in f:\n            line = line.strip().split("\t")\n            if len(line) != 2:\n                continue\n\n            # ANAPODA!!!!!!!  \n            hum_prot = line[1]\n            bac_prot = line[0]\n\n            if hum_prot not in hum_prot_degree:\n                hum_prot_degree[hum_prot] = 1\n            else:\n                hum_prot_degree[hum_prot] += 1\n\n            if bac_prot not in bac_prot_degree:\n                bac_prot_degree[bac_prot] = 1\n            else:\n                bac_prot_degree[bac_prot] += 1\n\n    # Save to csv\n    hum_prot_degree_df = pd.DataFrame.from_dict(hum_prot_degree, orient="index", columns=["degree_human"])\n    hum_prot_degree_df.sort_values("degree_human", ascending=Fals

Needed to do an id mapping in order to obtain higher degree proteins with non missing protein names and non missing GO: Either MF or BP. 

In [90]:
import os
import pandas as pd

with pd.ExcelWriter("./data/protein_degrees_by_phenotype.xlsx", engine="xlsxwriter") as writer:

    for ph in ["Healthy-associated", "CRC-associated"]:
        
        hum_degree_file = f"./data/{ph}_human_protein_degrees.csv"
        bac_degree_file = f"./data/{ph}_bacterial_protein_degrees.csv"

        hum_prot_degree = pd.read_csv(hum_degree_file, index_col=0)
        bac_prot_degree = pd.read_csv(bac_degree_file, index_col=0)

        bac_prot_degree = bac_prot_degree.reset_index()
        bac_prot_degree = bac_prot_degree.rename(columns={"index": "protein"})

        hum_prot_degree = hum_prot_degree.reset_index()
        hum_prot_degree = hum_prot_degree.rename(columns={"index": "protein"})

        bac_degree_mapping_file = f"./data/bacterial_{ph}_mapping.tsv.gz"
        hum_degree_mapping_file = f"./data/human_{ph}_mapping.tsv.gz"
        bac_degree_mapping = pd.read_csv(bac_degree_mapping_file, sep="\t", index_col=0) 
        # Drop nans of Protein names
        bac_degree_mapping = bac_degree_mapping.dropna(subset=["Protein names"])
        # Drop rows where both Gene Ontology (molecular function) and Gene Ontology (biological process) are NaN
        bac_degree_mapping = bac_degree_mapping.dropna(subset=["Gene Ontology (molecular function)", "Gene Ontology (biological process)"], how='all')

        # Merge the human/bacterial protein degree with the mapping. bac/hum_prot_degree protein column and bac/hum_degree_mapping Entry column. Keep columns from both. hum_degree_mapping had fiewer rows
        bac_prot_degree = bac_prot_degree.merge(bac_degree_mapping, left_on="protein", right_on="Entry", how="inner")
        bac_prot_degree = bac_prot_degree.drop(columns=["Entry"])

        # Same process
        hum_degree_mapping = pd.read_csv(hum_degree_mapping_file, sep="\t", index_col=0)
        hum_degree_mapping = hum_degree_mapping.dropna(subset=["Protein names"])
        hum_degree_mapping = hum_degree_mapping.dropna(subset=["Gene Ontology (molecular function)", "Gene Ontology (biological process)"], how='all')
        hum_prot_degree = hum_prot_degree.merge(hum_degree_mapping, left_on="protein", right_on="Entry", how="inner")
        hum_prot_degree = hum_prot_degree.drop(columns=["Entry"])

        # On both dataframes change. protein to Protein, and Degree_human or Degree_bacterial to Degree
        hum_prot_degree = hum_prot_degree.rename(columns={"protein": "Protein", "degree_human": "Degree"})
        bac_prot_degree = bac_prot_degree.rename(columns={"protein": "Protein", "degree_bacterial": "Degree"})
        
        print(bac_prot_degree.sort_values("Degree", ascending=False))
        print(hum_prot_degree.sort_values("Degree", ascending=False))
        print("-" * 100)

        hum_prot_degree.to_excel(writer, sheet_name=f"{ph}_Human", index=False)
        bac_prot_degree.to_excel(writer, sheet_name=f"{ph}_Bacterial", index=False)

        Protein  Degree                                      Protein names  \
0    A0A1T5GC38   29343            Cellulase (Glycosyl hydrolase family 5)   
1    A0A1T5D2V2   29307                Sugar phosphate isomerase/epimerase   
2    A0A1T5GAW1   29307                 Thymidylate synthase (EC 2.1.1.45)   
3    A0A1T5BRU8   29307                        ATPase/GTPase, AAA15 family   
4    A0A1T5CV36   29280                GTP cyclohydrolase 1 type 2 homolog   
..          ...     ...                                                ...   
389      R5DXX6   19532                        Pilus assembly protein CpaF   
390      R5I5H8   19530  Right handed beta helix domain-containing protein   
391      R6Q9D7   19530                 Dockerin domain-containing protein   
392      Q8A583   19530                Formamidopyrimidine-DNA glycosylase   
393      R6QCU0   19530  Ribulose-5-phosphate reductase (Ribulose-5-P r...   

                    Gene Ontology (biological process)  \
0    

In [ ]:
'''crc_cooc_file = "./data/CRC_Co_occurance_filtered.csv"
crc_cooc = pd.read_csv(crc_cooc_file, index_col=0)
crc_cooc = crc_cooc.dropna().reset_index(drop=True)
crc_cooc["pValue"] = crc_cooc["pValue"].map(lambda x: f"{x:.3e}")
print("CRC Co-occurrence statistics:")
print(crc_cooc["NoHost"].describe())
print("Number of unique bacterial pairs:", len(crc_cooc))
crc_cooc'''

'crc_cooc_file = "./data/CRC_Co_occurance_filtered.csv"\ncrc_cooc = pd.read_csv(crc_cooc_file, index_col=0)\ncrc_cooc = crc_cooc.dropna().reset_index(drop=True)\ncrc_cooc["pValue"] = crc_cooc["pValue"].map(lambda x: f"{x:.3e}")\nprint("CRC Co-occurrence statistics:")\nprint(crc_cooc["NoHost"].describe())\nprint("Number of unique bacterial pairs:", len(crc_cooc))\ncrc_cooc'

In [ ]:
'''healthy_cooc_file = "./data/Healthy_Co_occurance_filtered.csv"
healthy_cooc = pd.read_csv(healthy_cooc_file, index_col=0)
healthy_cooc = healthy_cooc.dropna().reset_index(drop=True)
healthy_cooc["pValue"] = healthy_cooc["pValue"].map(lambda x: f"{x:.3e}")
print("Healthy Co-occurrence statistics:")
print(healthy_cooc["NoHost"].describe())
print("Number of unique bacterial pairs:", len(healthy_cooc))
healthy_cooc'''

'healthy_cooc_file = "./data/Healthy_Co_occurance_filtered.csv"\nhealthy_cooc = pd.read_csv(healthy_cooc_file, index_col=0)\nhealthy_cooc = healthy_cooc.dropna().reset_index(drop=True)\nhealthy_cooc["pValue"] = healthy_cooc["pValue"].map(lambda x: f"{x:.3e}")\nprint("Healthy Co-occurrence statistics:")\nprint(healthy_cooc["NoHost"].describe())\nprint("Number of unique bacterial pairs:", len(healthy_cooc))\nhealthy_cooc'